# 01 — X-ray absorption coefficients (NIST mass attenuation)

`midas_hkls.absorption` provides the photon attenuation a beam experiences
passing through a material:

- `mass_attenuation_coefficient(element, λ)` → σ_mass [cm²/g]
- `linear_absorption_coefficient(element, λ)` → μ [cm⁻¹] = ρ · σ_mass

Values come from the NIST XCOM table, interpolated log-log in energy. This
module is **numpy-only** — it always runs, no optional extras. It is also
torch-differentiable through the wavelength when you pass a tensor, which is
useful for absorption-correction refinement.

Units follow the package convention: wavelengths in Å, μ in cm⁻¹.


In [1]:
import numpy as np

from midas_hkls.absorption import (
    available_elements_absorption,
    atomic_mass,
    element_density,
    mass_attenuation_coefficient,
    linear_absorption_coefficient,
)

elems = available_elements_absorption()
print(f"{len(elems)} elements tabulated; e.g. "
      f"Ti M={atomic_mass('Ti'):.3f} g/mol, ρ={element_density('Ti')} g/cm³")


92 elements tabulated; e.g. Ti M=47.867 g/mol, ρ=4.506 g/cm³


## μ at a typical HEDM energy

CP-Ti at λ = 0.173 Å (≈ 71.7 keV). xraylib/NIST give μ/ρ ≈ 0.509 cm²/g, so
μ = ρ·σ_mass ≈ 2.30 cm⁻¹. Our log-log interpolation matches to ~1%.


In [2]:
lam = 0.173
sigma = mass_attenuation_coefficient("Ti", lam)
mu = linear_absorption_coefficient("Ti", lam)
print(f"σ_mass(Ti, {lam} Å) = {sigma:.4f} cm²/g")
print(f"μ(Ti, {lam} Å)      = {mu:.4f} cm⁻¹   (NIST ≈ 2.295)")
assert abs(mu - 2.295) < 0.05


σ_mass(Ti, 0.173 Å) = 0.5093 cm²/g
μ(Ti, 0.173 Å)      = 2.2951 cm⁻¹   (NIST ≈ 2.295)


## Energy dependence

Away from absorption edges, μ falls steeply with photon energy (rises with
wavelength). We sweep Ti across the HEDM band and confirm the monotonic trend.


In [3]:
energies_keV = np.array([100, 80, 60, 40, 20, 12], dtype=float)
lams = 12.398 / energies_keV                 # E[keV] = 12.398 / λ[Å]
mus = np.array([linear_absorption_coefficient("Ti", l) for l in lams])
for e, l, m in zip(energies_keV, lams, mus):
    print(f"  {e:5.0f} keV  (λ={l:.3f} Å)   μ = {m:8.3f} cm⁻¹")
# Larger λ (lower E) ⇒ larger μ
assert np.all(np.diff(mus) > 0)


    100 keV  (λ=0.124 Å)   μ =    1.226 cm⁻¹
     80 keV  (λ=0.155 Å)   μ =    1.826 cm⁻¹
     60 keV  (λ=0.207 Å)   μ =    3.452 cm⁻¹
     40 keV  (λ=0.310 Å)   μ =    9.969 cm⁻¹
     20 keV  (λ=0.620 Å)   μ =   71.424 cm⁻¹
     12 keV  (λ=1.033 Å)   μ =  301.029 cm⁻¹


## Density override

μ scales linearly with density. For porous or alloyed samples pass an explicit
`density_g_cm3` instead of the tabulated bulk value.


In [4]:
mu_full = linear_absorption_coefficient("Ti", lam)
mu_half = linear_absorption_coefficient("Ti", lam, density_g_cm3=element_density("Ti") / 2)
print(f"μ(full ρ) / μ(half ρ) = {mu_full / mu_half:.4f}  (expect 2.0)")
assert abs(mu_full / mu_half - 2.0) < 1e-6


μ(full ρ) / μ(half ρ) = 2.0000  (expect 2.0)


## Differentiable in wavelength (torch)

When λ is a torch tensor, μ carries a gradient — handy if absorption is part
of a larger differentiable model. (If torch is not installed, this cell is
skipped.)


In [5]:
try:
    import torch
    lam_t = torch.tensor(0.173, dtype=torch.float64, requires_grad=True)
    mu_t = linear_absorption_coefficient("Ti", lam_t)
    mu_t.backward()
    print(f"μ = {float(mu_t):.4f} cm⁻¹   dμ/dλ = {float(lam_t.grad):.3f} cm⁻¹/Å")
    assert torch.isfinite(lam_t.grad)
except ImportError:
    print("(torch not installed — differentiable path skipped)")


μ = 2.2951 cm⁻¹   dμ/dλ = 28.673 cm⁻¹/Å


/var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/ipykernel_58057/3048623883.py:6: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print(f"μ = {float(mu_t):.4f} cm⁻¹   dμ/dλ = {float(lam_t.grad):.3f} cm⁻¹/Å")


That is the whole absorption surface: a NIST-backed μ(element, λ) that is a
plain float in the numpy path and a differentiable tensor in the torch path,
with an optional density override for non-bulk samples.
